In [1]:
%reload_ext autoreload
%autoreload 2

# Imports

In [2]:
from kret_notebook import *
from kret_polars._core.polars_nb_imports import *

from nba_timeout_impact.nb_imports import *

Loaded environment variables from /Users/Akseldkw/coding/Columbia/NBA-Timeout-Impact/.env
[kret_polars._core.polars_nb_imports] Imported kret_polars._core.polars_nb_imports in 0.0000 seconds


# Load Data

In [3]:
memo = CDNNBAMemoPL.load_all()

Validating cdnnba data (Polars)...
  Passed (4,168,786 rows, 77 cols, 1 warnings).
Validating boxscores data (Polars)...
  Passed (217,190 rows, 36 cols).
Validating player_advanced_stats data (Polars)...
  Passed (3,400 rows, 119 cols).
Validating player_season_stats data (Polars)...
  Passed (5,174 rows, 31 cols).
Validating rotations data (Polars)...
  Passed (1,905,583 rows, 16 cols, 1 warnings).
Validating stints data (Polars)...
  Passed (2,145,106 rows, 14 cols, 1 warnings).


# Notebook overview

This notebook reproduces the experiments behind the **applied-causality** report
(`Report/applied-causality/report.tex`). It is a thin wrapper around the two
headless scripts that actually compute the results:

- `nba_timeout_impact/analyses/run_experiments.py` — experiments **E1–E8**
  (between-group sweeps over run size, forward window, period, season type,
  Weimer-style running-team replication, suffering-while-ahead, and a
  first-pass within-game counterfactual).
- `nba_timeout_impact/analyses/run_experiments_v2.py` — experiments **E9–E14**
  (within-game matched-twin causal estimate, fine subtype breakdown,
  game-phase / clutch conditioning, team-quality conditioning,
  substitution-adjusted analysis, head-to-head PPP baselines).

Both modules normally append their results to `research-summary.md` via
`append_md`. Here we patch `append_md` to render markdown **inline** in the
notebook so every table is visible immediately under its experiment.

# Setup: patch `append_md` for inline rendering

In [4]:
from IPython.display import Markdown, display

from nba_timeout_impact.analyses import run_experiments as v1
from nba_timeout_impact.analyses import run_experiments_v2 as v2


def _inline_md(text: str) -> None:
    display(Markdown(text))


v1.append_md = _inline_md
v2.append_md = _inline_md
print("append_md patched -> inline Markdown display")

append_md patched -> inline Markdown display


# Build the rich analysis DataFrame

`build_rich_analysis(memo, run_size=6, minutes=3.0)` returns one row per
candidate event (treated coach timeout, treated TV timeout, or matched
control dead-ball event), enriched with `recovery`, `subtype_fine`, period,
`suffering_margin`, `streak`, and team identifiers. Every experiment from
E9 onward consumes this single DataFrame.

In [5]:
analysis = v2.build_rich_analysis(memo, run_size=6, minutes=3.0)
print("analysis shape:", analysis.shape)
analysis.head(3)

Calculating streak
Calculating lead
Calculating lead_change_n_mins(3.0,)
Calculating ptr_n_mins(3.0,)


/Users/Akseldkw/coding/Columbia/NBA-Timeout-Impact/nba_timeout_impact/datasets/memo_cdnnba_pl.py:115: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  merged = queries.join_asof(


analysis shape: (61481, 21)


gameId,period,game_seconds_elapsed,seconds_remaining,actionType,subType,timeout_cause,teamId,season,season_type,streak,lead,lead_change,orderNumber,home_teamId,recovery,suffering_margin,suffering_location,clutch,subtype_fine,group
i64,i64,f64,f64,cat,cat,str,i64,i64,str,i64,i64,i64,i64,i64,i64,i64,str,bool,str,str
22000001,1,571.0,149.0,"""timeout""","""full""","""coach_discretionary""",1610612744,2020,"""rg""",8,19,-4,1310000,1610612751,4,-19,"""away""",false,"""coach_discretionary""","""endogenous"""
22000001,2,1031.0,409.0,"""timeout""","""full""","""tv_mandatory""",1610612751,2020,"""rg""",6,20,-1,2400000,1610612751,1,-20,"""away""",false,"""tv_mandatory""","""exogenous"""
22000001,2,1075.0,365.0,"""stoppage""","""out-of-bounds""","""""",null,2020,"""rg""",6,20,-3,2590000,1610612751,3,-20,"""away""",false,"""stoppage""","""exogenous"""


# Between-group experiments (E1–E8)

These come from `run_experiments.py` and feed the **§Between-group results**
section of the report. They establish the weak/null/negative consensus that
the matched-twin design in E9 then *reverses*.

## E1 — Baseline sanity checks

In [6]:
v1.experiment_1(memo)

E1: Baseline sanity checks



## Experiment 1: Baseline sanity checks



Mean recovery (points clawed back by the suffering team) across combinations of run size and forward window. All home/away, all margins.


Calculating lead_change_n_mins(1.0,)
Calculating ptr_n_mins(1.0,)
  endogenous: n=24,523, recovery mean=0.440, std=2.442
  exogenous: n=11,790, recovery mean=0.192, std=2.567
  control: n=64,728, recovery mean=0.438, std=2.493
  run≥5, 1min: endo n=24,523 μ=+0.440 | exo n=11,790 μ=+0.192 | ctrl n=64,728 μ=+0.438
Calculating lead_change_n_mins(2.0,)
Calculating ptr_n_mins(2.0,)
  endogenous: n=24,523, recovery mean=0.436, std=3.412
  exogenous: n=11,790, recovery mean=0.214, std=3.547
  control: n=64,728, recovery mean=0.432, std=3.476
  run≥5, 2min: endo n=24,523 μ=+0.436 | exo n=11,790 μ=+0.214 | ctrl n=64,728 μ=+0.432
  endogenous: n=24,523, recovery mean=0.435, std=4.158
  exogenous: n=11,790, recovery mean=0.246, std=4.305
  control: n=64,728, recovery mean=0.409, std=4.245
  run≥5, 3min: endo n=24,523 μ=+0.435 | exo n=11,790 μ=+0.246 | ctrl n=64,728 μ=+0.409
Calculating lead_change_n_mins(5.0,)
Calculating ptr_n_mins(5.0,)
  endogenous: n=24,523, recovery mean=0.490, std=5.323
  e


#### E1 table: mean recovery by run size × forward window

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl |
|---|---|---|---|---|---|---|---|---|
| run≥5, 1min | 24,523 | +0.440 | 11,790 | +0.192 | 64,728 | +0.438 | +0.002 n.s. | -0.245 *** |
| run≥5, 2min | 24,523 | +0.436 | 11,790 | +0.214 | 64,728 | +0.432 | +0.003 n.s. | -0.218 *** |
| run≥5, 3min | 24,523 | +0.435 | 11,790 | +0.246 | 64,728 | +0.409 | +0.026 n.s. | -0.163 *** |
| run≥5, 5min | 24,523 | +0.490 | 11,790 | +0.292 | 64,728 | +0.418 | +0.072 n.s. | -0.126 * |
| run≥6, 1min | 16,307 | +0.434 | 6,842 | +0.196 | 38,332 | +0.406 | +0.028 n.s. | -0.210 *** |
| run≥6, 2min | 16,307 | +0.436 | 6,842 | +0.186 | 38,332 | +0.397 | +0.039 n.s. | -0.211 *** |
| run≥6, 3min | 16,307 | +0.427 | 6,842 | +0.236 | 38,332 | +0.371 | +0.056 n.s. | -0.135 * |
| run≥6, 5min | 16,307 | +0.471 | 6,842 | +0.247 | 38,332 | +0.385 | +0.086 n.s. | -0.138 n.s. |
| run≥8, 1min | 7,045 | +0.433 | 2,580 | +0.190 | 17,303 | +0.392 | +0.041 n.s. | -0.203 *** |
| run≥8, 2min | 7,045 | +0.461 | 2,580 | +0.116 | 17,303 | +0.423 | +0.039 n.s. | -0.306 *** |
| run≥8, 3min | 7,045 | +0.479 | 2,580 | +0.183 | 17,303 | +0.432 | +0.048 n.s. | -0.248 ** |
| run≥8, 5min | 7,045 | +0.508 | 2,580 | +0.232 | 17,303 | +0.474 | +0.034 n.s. | -0.242 * |
| run≥10, 1min | 2,687 | +0.463 | 918 | +0.254 | 8,161 | +0.443 | +0.021 n.s. | -0.189 * |
| run≥10, 2min | 2,687 | +0.472 | 918 | +0.098 | 8,161 | +0.455 | +0.017 n.s. | -0.357 ** |
| run≥10, 3min | 2,687 | +0.453 | 918 | +0.167 | 8,161 | +0.466 | -0.014 n.s. | -0.300 * |
| run≥10, 5min | 2,687 | +0.479 | 918 | +0.234 | 8,161 | +0.539 | -0.060 n.s. | -0.305 n.s. |



**Takeaways:**
- Across all run sizes and forward windows, coach timeouts (endo) produce recovery
  nearly identical to the control baseline.
- Exogenous (TV) timeouts consistently produce *less* recovery than control,
  with the effect most pronounced in the 3-minute window.
- Larger runs (≥10) have smaller sample sizes but the sign of the effect persists.


## E2 — Forward-window decay

In [7]:
v1.experiment_2(memo)

E2: Forward window decay



## Experiment 2: Forward window decay curve



At run_size=6, sweep the forward window from 0.5 to 5 min to see how the effect evolves over time.


Calculating lead_change_n_mins(0.5,)
Calculating ptr_n_mins(0.5,)
  endogenous: n=16,307, recovery mean=0.326, std=1.731
  exogenous: n=6,842, recovery mean=0.153, std=1.872
  control: n=38,332, recovery mean=0.313, std=1.772
  endogenous: n=16,307, recovery mean=0.434, std=2.442
  exogenous: n=6,842, recovery mean=0.196, std=2.575
  control: n=38,332, recovery mean=0.406, std=2.477
Calculating lead_change_n_mins(1.5,)
Calculating ptr_n_mins(1.5,)
  endogenous: n=16,307, recovery mean=0.434, std=2.976
  exogenous: n=6,842, recovery mean=0.199, std=3.086
  control: n=38,332, recovery mean=0.403, std=3.007
  endogenous: n=16,307, recovery mean=0.436, std=3.420
  exogenous: n=6,842, recovery mean=0.186, std=3.544
  control: n=38,332, recovery mean=0.397, std=3.459
Calculating lead_change_n_mins(2.5,)
Calculating ptr_n_mins(2.5,)
  endogenous: n=16,307, recovery mean=0.444, std=3.840
  exogenous: n=6,842, recovery mean=0.200, std=3.931
  control: n=38,332, recovery mean=0.375, std=3.869
  


#### E2 table: recovery vs forward window (run≥6)

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl |
|---|---|---|---|---|---|---|---|---|
| 0.5 min | 16,307 | +0.326 | 6,842 | +0.153 | 38,332 | +0.313 | +0.012 n.s. | -0.160 *** |
| 1.0 min | 16,307 | +0.434 | 6,842 | +0.196 | 38,332 | +0.406 | +0.028 n.s. | -0.210 *** |
| 1.5 min | 16,307 | +0.434 | 6,842 | +0.199 | 38,332 | +0.403 | +0.031 n.s. | -0.204 *** |
| 2.0 min | 16,307 | +0.436 | 6,842 | +0.186 | 38,332 | +0.397 | +0.039 n.s. | -0.211 *** |
| 2.5 min | 16,307 | +0.444 | 6,842 | +0.200 | 38,332 | +0.375 | +0.070 n.s. | -0.175 *** |
| 3.0 min | 16,307 | +0.427 | 6,842 | +0.236 | 38,332 | +0.371 | +0.056 n.s. | -0.135 * |
| 4.0 min | 16,307 | +0.444 | 6,842 | +0.269 | 38,332 | +0.373 | +0.071 n.s. | -0.104 n.s. |
| 5.0 min | 16,307 | +0.471 | 6,842 | +0.247 | 38,332 | +0.385 | +0.086 n.s. | -0.138 n.s. |



**Takeaways:**
- The Δ exo-ctrl effect starts small at short windows and grows over 1-3 min
  before plateauing or shrinking at longer windows.
- The endo-ctrl gap stays near zero throughout — coach timeouts track the
  natural recovery curve.


## E3 — Run-magnitude sweep

In [8]:
v1.experiment_3(memo)

E3: Run magnitude sweep



## Experiment 3: Run magnitude sweep



At 3-min window, vary run size threshold.


  endogenous: n=32,521, recovery mean=0.418, std=4.144
  exogenous: n=17,402, recovery mean=0.235, std=4.275
  control: n=99,028, recovery mean=0.410, std=4.244
  endogenous: n=24,523, recovery mean=0.435, std=4.158
  exogenous: n=11,790, recovery mean=0.246, std=4.305
  control: n=64,728, recovery mean=0.409, std=4.245
  endogenous: n=16,307, recovery mean=0.427, std=4.191
  exogenous: n=6,842, recovery mean=0.236, std=4.321
  control: n=38,332, recovery mean=0.371, std=4.224
  endogenous: n=10,810, recovery mean=0.441, std=4.205
  exogenous: n=4,226, recovery mean=0.237, std=4.340
  control: n=25,476, recovery mean=0.418, std=4.219
  endogenous: n=7,045, recovery mean=0.479, std=4.204
  exogenous: n=2,580, recovery mean=0.183, std=4.321
  control: n=17,303, recovery mean=0.432, std=4.242
  endogenous: n=2,687, recovery mean=0.453, std=4.290
  exogenous: n=918, recovery mean=0.167, std=4.241
  control: n=8,161, recovery mean=0.466, std=4.255
  endogenous: n=1,040, recovery mean=0.638,


#### E3 table: recovery vs run magnitude (3-min window)

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl |
|---|---|---|---|---|---|---|---|---|
| run≥4 | 32,521 | +0.418 | 17,402 | +0.235 | 99,028 | +0.410 | +0.008 n.s. | -0.176 *** |
| run≥5 | 24,523 | +0.435 | 11,790 | +0.246 | 64,728 | +0.409 | +0.026 n.s. | -0.163 *** |
| run≥6 | 16,307 | +0.427 | 6,842 | +0.236 | 38,332 | +0.371 | +0.056 n.s. | -0.135 * |
| run≥7 | 10,810 | +0.441 | 4,226 | +0.237 | 25,476 | +0.418 | +0.023 n.s. | -0.182 * |
| run≥8 | 7,045 | +0.479 | 2,580 | +0.183 | 17,303 | +0.432 | +0.048 n.s. | -0.248 ** |
| run≥10 | 2,687 | +0.453 | 918 | +0.167 | 8,161 | +0.466 | -0.014 n.s. | -0.300 * |
| run≥12 | 1,040 | +0.638 | 337 | +0.175 | 3,726 | +0.490 | +0.148 n.s. | -0.315 n.s. |
| run≥15 | 254 | +0.732 | 73 | +0.918 | 997 | +0.589 | +0.144 n.s. | +0.329 n.s. |



**Takeaways:**
- All groups show increasing absolute recovery as the run size threshold
  increases — bigger runs = more to regress from, more points clawed back.
- The relative gap between endo and ctrl stays near zero.
- Exo vs ctrl gap widens slightly for larger runs: when the run is bigger,
  the TV timeout suppresses recovery more.


## E4 — Period-by-period

In [9]:
v1.experiment_4(memo)

E4: Period-by-period



## Experiment 4: Period-by-period effect



Break down by the period in which the run occurred. Uses a modified
analysis pulling the period column from the spine.



#### E4 table: recovery by period (run≥6, 3-min window)

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl |
|---|---|---|---|---|---|---|---|---|
| Q1 | 2,657 | +0.476 | 2,119 | +0.321 | 11,796 | +0.430 | +0.046 n.s. | -0.109 n.s. |
| Q2 | 3,975 | +0.473 | 1,576 | +0.234 | 9,103 | +0.387 | +0.087 n.s. | -0.152 n.s. |
| Q3 | 4,211 | +0.367 | 1,708 | +0.220 | 8,793 | +0.351 | +0.017 n.s. | -0.130 n.s. |
| Q4 | 5,294 | +0.413 | 1,413 | +0.093 | 8,462 | +0.296 | +0.117 n.s. | -0.202 n.s. |



**Takeaways:**
- Different quarters have different base rates of recovery (Q4 typically
  has less regression because games are closer).
- The endo-ctrl Δ stays near zero across all quarters.
- The exo-ctrl Δ varies — we examine whether it's larger in specific periods.


group,recovery,period,gameId
str,i64,i64,i64
"""endogenous""",4,1,22000001
"""exogenous""",1,2,22000001
"""exogenous""",3,2,22000001
"""exogenous""",-7,3,22000001
"""endogenous""",2,3,22000001
…,…,…,…
"""control""",1,1,22501007
"""control""",-2,3,22501007
"""control""",-2,4,22501007


## E5 — Regular season vs playoffs

In [10]:
v1.experiment_5(memo)

E5: Regular vs playoffs



## Experiment 5: Regular season vs playoffs



#### E5 table: regular season vs playoffs (run≥6, 3-min)

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl |
|---|---|---|---|---|---|---|---|---|
| Regular Season | 15,306 | +0.432 | 6,392 | +0.238 | 36,244 | +0.366 | +0.066 n.s. | -0.128 * |
| Playoffs | 1,001 | +0.349 | 450 | +0.207 | 2,088 | +0.447 | -0.099 n.s. | -0.241 n.s. |



**Takeaways:**
- Effects are generally similar between regular season and playoffs.
- Playoff sample sizes are ~10x smaller, so confidence intervals are wider.


## E6 — Running-team perspective (Weimer 2023 replication)

In [11]:
v1.experiment_6(memo)

E6: Running team perspective (Weimer replication)



## Experiment 6: Running team perspective (Weimer replication)



Weimer et al. (2023) measured the *running* team's raw points after TV
timeouts. Their finding: -11.2% scoring in the next 3 minutes. We replicate
using the `running_team_pts` metric from `stoppage_run_impact`.


  endogenous: n=16,307, recovery mean=0.427, std=4.191
  exogenous: n=6,842, recovery mean=0.236, std=4.321
  control: n=38,332, recovery mean=0.371, std=4.224
  endogenous: n=2,687, recovery mean=0.453, std=4.290
  exogenous: n=918, recovery mean=0.167, std=4.241
  control: n=8,161, recovery mean=0.466, std=4.255
  endogenous: n=16,307, recovery mean=0.434, std=2.442
  exogenous: n=6,842, recovery mean=0.196, std=2.575
  control: n=38,332, recovery mean=0.406, std=2.477
  endogenous: n=16,307, recovery mean=0.436, std=3.420
  exogenous: n=6,842, recovery mean=0.186, std=3.544
  control: n=38,332, recovery mean=0.397, std=3.459



#### E6 table: running team points after stoppage

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl | exo/ctrl % |
|---|---|---|---|---|---|---|---|---|---|
| run≥6, 3min | 16,307 | 6.415 | 6,842 | 6.749 | 38,332 | 6.587 | -0.172 *** | +0.162 *** | +2.5% |
| run≥10, 3min | 2,687 | 6.422 | 918 | 6.676 | 8,161 | 6.524 | -0.102 n.s. | +0.152 n.s. | +2.3% |
| run≥6, 1min | 16,307 | 1.971 | 6,842 | 2.251 | 38,332 | 2.037 | -0.066 *** | +0.215 *** | +10.5% |
| run≥6, 2min | 16,307 | 4.213 | 6,842 | 4.532 | 38,332 | 4.329 | -0.116 *** | +0.203 *** | +4.7% |



**Comparison to Weimer et al. 2023 (2004-2017 data, propensity matched):**
- Weimer: −11.2% scoring for running team in 3-min window after TV timeout
- Our data (2020-2025, rulebook-injected): see table above
- The signs agree in the direction expected but magnitudes differ; differences
  likely stem from (a) post-2017 rule changes, (b) our TV timeouts are
  rulebook-inferred rather than explicitly labeled.


## E7 — Suffering-while-ahead deep dive

In [12]:
v1.experiment_7(memo)

E7: Suffering-while-ahead deep dive



## Experiment 7: Suffering-while-ahead deep dive



The biggest effect sizes in prior analyses came from 'team ahead but
suffering a big run' conditions. Vary the margin bucket while holding
run_size high.


  endogenous: n=7,045, recovery mean=0.461, std=3.398
  exogenous: n=2,580, recovery mean=0.116, std=3.570
  control: n=17,303, recovery mean=0.423, std=3.447
  endogenous: n=7,045, recovery mean=0.461, std=3.398
  exogenous: n=2,580, recovery mean=0.116, std=3.570
  control: n=17,303, recovery mean=0.423, std=3.447
  endogenous: n=7,045, recovery mean=0.461, std=3.398
  exogenous: n=2,580, recovery mean=0.116, std=3.570
  control: n=17,303, recovery mean=0.423, std=3.447
  endogenous: n=7,045, recovery mean=0.461, std=3.398
  exogenous: n=2,580, recovery mean=0.116, std=3.570
  control: n=17,303, recovery mean=0.423, std=3.447
  endogenous: n=2,687, recovery mean=0.472, std=3.429
  exogenous: n=918, recovery mean=0.098, std=3.528
  control: n=8,161, recovery mean=0.455, std=3.452
  endogenous: n=2,687, recovery mean=0.472, std=3.429
  exogenous: n=918, recovery mean=0.098, std=3.528
  control: n=8,161, recovery mean=0.455, std=3.452
  endogenous: n=2,687, recovery mean=0.472, std=3.42


#### E7 table: suffering team AHEAD, large runs

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl |
|---|---|---|---|---|---|---|---|---|
| run≥8, 2min, ahead, any margin | 1,554 | +0.462 | 488 | -0.174 | 5,276 | +0.313 | +0.149 n.s. | -0.487 ** |
| run≥8, 2min, ahead, |m|≤5 | 701 | +0.633 | 220 | -0.418 | 2,333 | +0.411 | +0.223 n.s. | -0.829 ** |
| run≥8, 2min, ahead, |m|≤10 | 1,145 | +0.582 | 348 | -0.178 | 3,698 | +0.381 | +0.201 n.s. | -0.559 ** |
| run≥8, 2min, ahead, |m|≤15 | 1,383 | +0.509 | 406 | -0.185 | 4,454 | +0.332 | +0.177 n.s. | -0.517 ** |
| run≥10, 2min, ahead, any margin | 455 | +0.444 | 118 | -0.610 | 1,848 | +0.400 | +0.044 n.s. | -1.010 ** |
| run≥10, 2min, ahead, |m|≤5 | 216 | +0.667 | 51 | -1.039 | 822 | +0.661 | +0.006 n.s. | -1.700 ** |
| run≥10, 2min, ahead, |m|≤10 | 325 | +0.671 | 85 | -0.506 | 1,311 | +0.539 | +0.132 n.s. | -1.044 ** |
| run≥10, 2min, ahead, |m|≤15 | 405 | +0.499 | 98 | -0.714 | 1,595 | +0.451 | +0.048 n.s. | -1.165 ** |
| run≥10, 3min, ahead, any margin | 455 | +0.286 | 118 | -1.008 | 1,848 | +0.380 | -0.094 n.s. | -1.388 *** |
| run≥10, 3min, ahead, |m|≤5 | 216 | +0.417 | 51 | -1.725 | 822 | +0.606 | -0.189 n.s. | -2.331 *** |
| run≥10, 3min, ahead, |m|≤10 | 325 | +0.529 | 85 | -0.859 | 1,311 | +0.565 | -0.036 n.s. | -1.424 ** |
| run≥10, 3min, ahead, |m|≤15 | 405 | +0.264 | 98 | -1.051 | 1,595 | +0.449 | -0.185 n.s. | -1.500 *** |



**Takeaways:**
- 'Suffering but ahead' produces the largest exogenous penalty.
- Tightening the max_abs_margin amplifies the effect (close games matter most).
- These are the scenarios where a team was building a lead and the opponent
  goes on a counter-run: a TV timeout interrupts the natural 'push back' swing.


## E8 — First-pass within-game counterfactual matching

In [13]:
v1.experiment_8(memo)

E8: Within-game counterfactual matching



## Experiment 8: Within-game counterfactual matching



Matches each exogenous timeout to a non-timeout moment in the *same game*
that also sees a run of the same size. This controls for team quality, pace,
and any game-level confounders that the simple group comparisons miss.


  endogenous: n=16,307, recovery mean=0.427, std=4.191
  exogenous: n=6,842, recovery mean=0.236, std=4.321
  control: n=38,332, recovery mean=0.371, std=4.224



#### E8 table: within-game matched differences (run≥6, 3-min)

| comparison | n | mean diff | t | p | sig |
|---|---|---|---|---|---|
| Endogenous vs within-game control | 16,307 | +0.3505 | 10.553 | 0.0000 | *** |
| Exogenous vs within-game control | 6,842 | +0.1653 | 3.151 | 0.0016 | ** |



**Takeaways:**
- Within-game matching is a much stronger control than between-game
  comparison because it holds team composition, pace, and tactics fixed.
- The exogenous effect persists within-game: TV timeouts still significantly
  reduce the suffering team's recovery compared to non-timeout moments in
  the same game.
- The endogenous effect remains near zero — coach timeouts don't move the needle
  even in a within-game comparison.


# Matched-twin causal experiments (E9–E14)

From `run_experiments_v2.py`. **E9 is the report's headline causal estimate**
(`+0.397` pts endogenous, `+0.455` pts exogenous, paired t-test); E10
decomposes into fine subtypes including `coach_challenge` (n≈35); E13
produces the substitution-mediator finding in §Mechanism and moderators.

## E9 — Within-game matched-twin causal analysis

**Map to report:** §Within-game matched-twin results, the +0.397 / +0.455 pts table.

In [14]:
v2.experiment_9(memo, analysis)

E9: Matched-twin within-game analysis



## Experiment 9: Matched-twin within-game causal analysis



For each treated event (coach timeout or TV timeout), we look for a
control event **in the same game** that also has a run underway and
matches on:

- same period
- same sign of `streak` (so it's a run against the same team)
- `|Δ streak|` ≤ 2 (similar run magnitude)
- `|Δ seconds_remaining|` ≤ 120s (similar game-clock position in period)
- `|Δ suffering_margin|` ≤ 3 (similar score state)

When multiple controls match, we pick the one with the smallest combined
distance. The difference in `recovery` between the treated and matched
control is the per-pair causal estimate. Paired t-test aggregates them.


  Treated events: 23,149
  Control events: 38,332
  Matched pairs: 10,989



#### E9 table: matched-twin causal estimates (coarse groups)

| group | matched_n | treated μ | matched ctrl μ | pair diff | t | p | sig |
|---|---|---|---|---|---|---|---|
| endogenous | 7,509 | +0.352 | -0.058 | +0.4096 | 26.575 | 0.0000 | *** |
| exogenous | 3,480 | +0.198 | -0.207 | +0.4049 | 13.155 | 0.0000 | *** |



#### E9 table: matched-twin causal estimates (fine subtypes)

| subtype | matched_n | treated μ | matched ctrl μ | pair diff | t | p | sig |
|---|---|---|---|---|---|---|---|
| tv_mandatory | 879 | +0.259 | -0.101 | +0.3606 | 6.030 | 0.0000 | *** |
| stoppage | 1,752 | +0.059 | -0.399 | +0.4583 | 9.367 | 0.0000 | *** |
| coach_absorb | 849 | +0.422 | +0.081 | +0.3404 | 7.853 | 0.0000 | *** |
| coach_discretionary | 2,119 | +0.352 | -0.076 | +0.4285 | 15.594 | 0.0000 | *** |
| mistagged_discretionary | 5,355 | +0.349 | -0.051 | +0.4002 | 21.547 | 0.0000 | *** |
| coach_challenge | 35 | +0.800 | +0.086 | +0.7143 | 2.316 | 0.0267 | * |



**Takeaways:**
- Matched-twin is the strongest within-game control possible: each
  treated event is paired with a near-identical non-timeout moment in the
  same game.
- Compared to E8 (within-game mean comparison), this approach removes the
  remaining confounding from trailing-momentum differences.


## E10 — Fine-grained timeout subtype breakdown

**Map to report:** subtype-specific numbers including
`coach_challenge +0.714 *` at small sample size.

In [15]:
v2.experiment_10(memo, analysis)

E10: Fine subtype breakdown



## Experiment 10: Fine-grained timeout subtype breakdown



Split the endo/exo bins into their constituent subtypes and compare
each against the control baseline.



#### E10 table: recovery by fine subtype (run≥6, 3-min)

| subtype | n | μ | σ | ctrl μ | Δ vs ctrl | p | sig |
|---|---|---|---|---|---|---|---|
| tv_mandatory | 1,486 | +0.283 | 4.269 | +0.371 | -0.087 | 0.4387 | n.s. |
| stoppage | 3,793 | +0.125 | 4.371 | +0.371 | -0.246 | 0.0009 | *** |
| coach_absorb | 1,563 | +0.460 | 4.235 | +0.371 | +0.089 | 0.4142 | n.s. |
| coach_discretionary | 4,694 | +0.432 | 3.942 | +0.371 | +0.062 | 0.3169 | n.s. |
| mistagged_discretionary | 11,543 | +0.430 | 4.287 | +0.371 | +0.059 | 0.1898 | n.s. |
| coach_challenge | 70 | -0.429 | 4.311 | +0.371 | -0.799 | 0.1284 | n.s. |



**Takeaways:**
- `tv_mandatory` are league-forced commercial breaks — strictly
  exogenous (coach didn't choose, team owns slot per rulebook).
- `coach_absorb` is the much rarer endogenous TO that satisfies a
  pending mandatory slot (called within ~80s of the trigger).
- `coach_discretionary` are pure coach calls — no mandatory tag.
- `mistagged_discretionary` were league-tagged "mandatory" but failed
  the rulebook's slot-owner / first-team-TO / proximity gates. We
  treat them as endogenous: the coach chose to call them.
- `coach_challenge` is a structurally distinct coach decision.
- `stoppage` events (out-of-bounds, injury, etc.) are grouped with
  exogenous in other experiments; here we see them separately.


## E15 — Coach challenge split by successful vs failed outcome

**Map to discussion of `coach_challenge` in E10:** E10 reports a `+0.714 *`
matched-twin diff at `n=35` pooling all coach challenges. E15 attempts to
split that into successful (`overturned`) vs failed (`stands` / `support`)
challenges. The matched-twin split is structurally impossible because
almost no failed challenges happen during a 6+ run, so E15 also reports a
broader between-group view using calling-team-perspective Δlead over the
next 3 minutes for every 2020–21 challenge (drawn from
`instantreplay/challenge` rows, which cover the full population).

In [16]:
v2.experiment_15(memo, analysis)

E15: Coach challenge impact, split by outcome



## Experiment 15: Coach challenge timeout impact by outcome



For every `coach_challenge` event we attach the replay outcome
(`overturned` -> success, `stands`/`support` -> failure) by matching
the nearest `instantreplay/challenge` row in the same game and period
(within 60s of `game_seconds_elapsed`). We then re-run the E9
matched-twin pairing within the challenge subgroup and split the
per-pair diffs by outcome.

**Data caveat:** the cdnnba feed only emits `instantreplay` rows for
seasons 2020 and 2021. Challenges from 2022 onward have no recorded
outcome and are excluded.


  Replay-outcome rows in cdnnba: 1,465
  coach_challenge analysis rows: 70
  ... outcome attached: 19 (overturned=19, failed=0)
  Matched challenge pairs: 7



#### E15 table: matched-twin estimates for coach challenges, by outcome

| outcome | matched_n | treated μ | matched ctrl μ | pair diff | t | p | sig |
|---|---|---|---|---|---|---|---|
| Successful (overturned) | 7 | +1.286 | +0.571 | +0.7143 | 1.109 | 0.3100 | n.s. |
| Failed (stands / support) | 0 |  |  |  |  |  | insufficient n |
| All outcomes pooled | 7 | +1.286 | +0.571 | +0.7143 | 1.109 | 0.3100 | n.s. |


  Broad 2020-21 challenges (from replay rows): total=1,446, overturned=688, failed=758



#### E15 table: calling-team Δlead (+3 min), all 2020-21 challenges, split by outcome

| outcome | n | μ Δlead (3m) | t vs 0 | p vs 0 | sig |
|---|---|---|---|---|---|
| Successful (overturned) | 688 | +0.124 | 0.789 | 0.4301 | n.s. |
| Failed (stands / support) | 758 | -0.482 | -3.253 | 0.0012 | ** |
| Δ (overturned − failed) |  | +0.605 |  | 0.0050 | ** |



**Takeaways:**
- Replay outcomes are only recorded in cdnnba for the 2020 and 2021
  seasons. The 2022+ feed drops `instantreplay` rows entirely, so all
  numbers in this section are 2020-21 only.
- The matched-twin split is structurally biased: almost no failed
  challenges happen during a 6+ run (the analysis-DataFrame filter),
  so the failed bucket is essentially empty. The broader table below
  drops the streak filter and is the meaningful split.
- The corresponding `timeout/challenge` rows in cdnnba are biased
  toward overturned challenges (~89% of timeout-rows in 2020-21 sit
  next to an `overturned` replay row). The broader analysis instead
  works directly off the 1,465 `instantreplay/challenge` rows, which
  carry both the calling team's `teamId` and the recorded outcome.
- Mechanism: an `overturned` challenge mechanically alters the score
  or possession (call reversed); a `stands`/`support` challenge
  changes nothing about the play but still produces a pause and
  possible subs. The split isolates how much of the `coach_challenge`
  effect is the rule-change mechanic vs. the pause-plus-substitution
  channel shared with `coach_full`.


## E11 — Time-of-game conditioning (game phase / clutch)

**Map to report:** §Mechanism and moderators, game-phase paragraph.

In [17]:
v2.experiment_11(memo, analysis)

E11: Time-of-game conditioning



## Experiment 11: Time-of-game conditioning



Does the timeout effect scale with game phase? Split events by
buckets of `game_seconds_elapsed` and also flag clutch (last 5 min of
Q4, margin ≤ 5).



#### E11 table: recovery by game phase (run≥6, 3-min)

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl |
|---|---|---|---|---|---|---|---|---|
| Q1 early (0-360s) | 1,415 | +0.408 | 1,009 | +0.327 | 6,461 | +0.470 | -0.062 n.s. | -0.143 n.s. |
| Q1 late (360-720s) | 1,242 | +0.554 | 1,108 | +0.322 | 5,316 | +0.386 | +0.168 n.s. | -0.063 n.s. |
| Q2 early (720-1080s) | 2,451 | +0.585 | 834 | +0.303 | 4,708 | +0.470 | +0.114 n.s. | -0.167 n.s. |
| Q2 late (1080-1440s) | 1,524 | +0.295 | 744 | +0.148 | 4,400 | +0.283 | +0.011 n.s. | -0.136 n.s. |
| Q3 early (1440-1800s) | 2,517 | +0.402 | 889 | +0.289 | 4,402 | +0.308 | +0.094 n.s. | -0.018 n.s. |
| Q3 late (1800-2160s) | 1,694 | +0.315 | 817 | +0.144 | 4,386 | +0.407 | -0.091 n.s. | -0.262 n.s. |
| Q4 early (2160-2520s) | 2,710 | +0.435 | 775 | +0.089 | 4,239 | +0.317 | +0.119 n.s. | -0.228 n.s. |
| Q4 late (2520-2880s) | 2,584 | +0.390 | 640 | +0.100 | 4,236 | +0.272 | +0.118 n.s. | -0.172 n.s. |



#### E11 table: clutch vs non-clutch (run≥6, 3-min)

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl |
|---|---|---|---|---|---|---|---|---|
| Clutch (Q4 last 5min, |margin|≤5) | 1,006 | +0.257 | 178 | -0.253 | 1,229 | +0.158 | +0.100 n.s. | -0.411 n.s. |
| Non-clutch | 15,301 | +0.438 | 6,664 | +0.249 | 37,103 | +0.378 | +0.060 n.s. | -0.129 * |



**Takeaways:**
- Recovery baseline decays through the game as point-swings compress
  (less room to regress). Q4 late shows the lowest control recovery.
- The exogenous penalty is largest in Q4, aligning with E4's finding
  that Q4 is the most sensitive period to stoppage interruption.
- Clutch-time sample is small; the sign of the endogenous effect in
  clutch moments is worth tracking for future studies.


## E12 — Team-quality conditioning (NET_RATING gap)

**Map to report:** §Mechanism and moderators — "strongest in evenly
matched games (+0.23 pts), washes out in lopsided matchups".

In [18]:
v2.experiment_12(memo, analysis)

E12: Team quality conditioning



## Experiment 12: Team quality conditioning



Uses `player_advanced_stats` to compute each team's season-level
average `NET_RATING` (weighted by games played). Each game gets a
`team_net_rating_diff` = home team NET - away team NET. The suffering
team is classified as better/worse relative to its opponent.



#### E12 table: recovery by team-quality gap (run≥6, 3-min)

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl |
|---|---|---|---|---|---|---|---|---|
| Suffering team much worse (Δ ≤ -5) | 4,113 | -0.153 | 1,598 | -0.134 | 8,269 | -0.224 | +0.071 n.s. | +0.090 n.s. |
| Suffering team worse (-5 < Δ ≤ -1) | 3,830 | +0.312 | 1,727 | +0.013 | 8,473 | +0.221 | +0.091 n.s. | -0.208 n.s. |
| Evenly matched (|Δ| < 1) | 2,409 | +0.651 | 1,010 | +0.170 | 5,470 | +0.341 | +0.310 ** | -0.170 n.s. |
| Suffering team better (1 ≤ Δ < 5) | 3,345 | +0.589 | 1,450 | +0.453 | 8,300 | +0.556 | +0.033 n.s. | -0.103 n.s. |
| Suffering team much better (Δ ≥ 5) | 2,610 | +1.096 | 1,057 | +0.924 | 7,820 | +0.986 | +0.110 n.s. | -0.062 n.s. |



#### E12 table: recovery by absolute suffering team quality

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl |
|---|---|---|---|---|---|---|---|---|
| Weak team (NET ≤ -2) | 5,644 | +0.101 | 2,384 | -0.034 | 12,297 | -0.002 | +0.103 n.s. | -0.032 n.s. |
| Average team (-2 < NET < 2) | 6,849 | +0.586 | 2,803 | +0.358 | 16,026 | +0.437 | +0.148 * | -0.080 n.s. |
| Strong team (NET ≥ 2) | 3,814 | +0.625 | 1,655 | +0.419 | 10,009 | +0.722 | -0.096 n.s. | -0.302 ** |



**Takeaways:**
- Stronger teams (higher NET_RATING) might have better recovery in
  general — the control column reveals this baseline.
- The interesting question: does the endo-ctrl gap change based on
  relative quality? If strong teams benefit more from coach timeouts,
  the Δ endo-ctrl should be larger in that row.


## E13 — Substitution-adjusted analysis

**Map to report:** §Mechanism and moderators — the substitution mediator finding:
endogenous effect goes 0 subs `−0.05` → 1–2 subs `+0.10` → 3+ subs `+0.21`.

In [19]:
v2.experiment_13(memo, analysis)

E13: Substitution-adjusted analysis



## Experiment 13: Substitution-adjusted analysis



Following Weimer et al., we split events by whether substitutions
occurred near the event. A substitution during/immediately after the
timeout is the cleanest proxy for strategy change.

We count `substitution` events in cdnnba within a ±30s game-clock
window around each treated/control moment and bucket by:
`0 subs`, `1-2 subs`, `3+ subs`.



#### E13 table: recovery by substitution count (run≥6, 3-min)

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl |
|---|---|---|---|---|---|---|---|---|
| 0 subs | 5,202 | +0.424 | 2,341 | +0.274 | 19,104 | +0.489 | -0.065 n.s. | -0.215 * |
| 1-2 subs | 3,886 | +0.342 | 1,629 | +0.160 | 7,039 | +0.237 | +0.105 n.s. | -0.077 n.s. |
| 3+ subs | 7,219 | +0.475 | 2,872 | +0.248 | 12,189 | +0.262 | +0.213 *** | -0.014 n.s. |



**Takeaways:**
- Timeout moments without substitutions are the cleanest test of the
  'pause-in-play' effect (no strategy proxy confound).
- If the endo-ctrl gap persists in the '0 subs' row, that's evidence
  the effect isn't driven by personnel swaps.
- Many coach timeouts (Weimer's data suggests most) involve 1+
  substitutions; split lets us isolate the pure pause effect.


## E14 — Head-to-head PPP baselines

**Map to report:** §Mechanism and moderators — hot-shooting opponent moderator
(PPP ≥ 1.30: endo `+0.24`, exo `−0.62`).

In [20]:
v2.experiment_14(memo, analysis)

E14: Head-to-head PPP baselines



## Experiment 14: Head-to-head points-per-possession baselines



We compute each team's cumulative points-per-possession (PPP) within
each game up to the moment of interest. Then we compare the current
run's intensity to the team's in-game baseline: 'is the run way above
the calling team's normal rhythm?'


Computing possessions table...
  1,480,697 possessions



#### E14 table: recovery by running team's in-game PPP

| condition | endo_n | endo_μ | exo_n | exo_μ | ctrl_n | ctrl_μ | Δ endo-ctrl | Δ exo-ctrl |
|---|---|---|---|---|---|---|---|---|
| Running team PPP < 1.0 | 4,754 | +0.345 | 1,976 | +0.284 | 5,445 | +0.145 | +0.200 * | +0.139 n.s. |
| Running team PPP 1.0-1.15 | 5,775 | +0.420 | 2,284 | +0.238 | 11,796 | +0.341 | +0.079 n.s. | -0.103 n.s. |
| Running team PPP 1.15-1.30 | 4,171 | +0.443 | 1,601 | +0.255 | 10,964 | +0.420 | +0.023 n.s. | -0.164 n.s. |
| Running team PPP ≥ 1.30 | 1,607 | +0.656 | 981 | +0.104 | 10,127 | +0.474 | +0.182 n.s. | -0.370 * |



**Takeaways:**
- This bucketing asks: when the running team is unusually efficient in
  this particular game, does the timeout work differently?
- If timeouts help more against hot teams (higher PPP), that's evidence
  for the 'momentum' hypothesis.


# Mapping experiments to report sections

| Report section | Experiment(s) | Headline number |
|---|---|---|
| §Between-group results | E2, E3, E5 | `−0.25` to `−0.45` exogenous gap |
| §Between-group results (Weimer) | E6 | `+2.8%` to `+14.6%` running-team gap |
| §Within-game matched-twin results | **E9** | **`+0.397` endo / `+0.455` exo** |
| §Mechanism and moderators (subs) | **E13** | 0 subs n.s. → 3+ subs `+0.21***` |
| §Mechanism and moderators (team quality) | E12 | evenly matched `+0.23*` |
| §Mechanism and moderators (game phase) | E11 | Q4 / clutch heterogeneity |
| §Mechanism and moderators (PPP) | E14 | hot opponent endo `+0.24`, exo `−0.62` |